In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import SGDRegressor
from sklearn.linear_model import HuberRegressor
from tqdm import tqdm
import statsmodels.api as sm

In [ ]:
import scipy as sp
from scipy import stats
# 导入绘图的库
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()
# 用于估计统计模型的库
import statsmodels.formula.api as smf
import statsmodels.api as sm

In [ ]:
with open("data/raw/characteristics_with_return.pkl", "rb") as f:
    data = pickle.load(f)

In [ ]:
data.head()

In [ ]:
def R2_cal(y, y_pred):
    r2 = 1 - np.sum((y-y_pred)**2) / np.sum(y**2)
    return r2

In [ ]:
# 60 years of data
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')

pred_y_list = []
true_y_list = []

# Loop over the years
for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')

    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')

    # Select the data for training and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    # Extract features and target variables
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
    y_train = train_data['RET'].values  
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values
    
    # 计算残差
    model = LinearRegression(fit_intercept=False)
    model.fit(X_train, y_train)
    residuals = np.abs(model.predict(X_train) - y_train)

    # 计算残差的99.9%分位数
    epsilon = np.percentile(residuals, 99.9)

    # 根据计算得到的epsilon值重新拟合模型
    model = SGDRegressor(fit_intercept=False, loss='huber', epsilon=epsilon)
    model.fit(X_train, y_train)
   
    # Predict on training and validation data
    y_test_pred = model.predict(X_test)

    pred_y_list.extend(y_test_pred)
    true_y_list.extend(y_test)

In [ ]:
# huber 损失
def huber(y_test, y_test_pred, delta):
    loss = np.where(np.abs(true - pred) < delta , 0.5*((true-pred)**2), delta*np.abs(true - pred) - 0.5*(delta**2))
    return np.sum(loss)

In [ ]:
test_result=pd.read_csv(r"data/raw/characteristics_with_return.pkl")
mod_glm=smf.glm('result~hours',data=test_result,family=sm.families.Binomial()).fit()